# 🌿 Seasonal NDVI Delta Calculation

## Purpose

This notebook calculates seasonal NDVI difference rasters between spring and summer vegetation rasters.

The workflow identifies areas where vegetation:

- decreased
- remained stable
- increased

between spring and late summer periods.

The output raster represents seasonal vegetation change.

---

## Input

Expected input rasters:

```text
NDVI_spring_2020_veg.tif
NDVI_summer_2020_veg.tif
```

Input folder:

```text
NDVI_filtered
```

---

## Delta Formula

```text
NDVI delta = summer NDVI - spring NDVI
```

Interpretation:

| Delta values | Meaning |
|---|---|
| Negative | vegetation decrease |
| Around 0 | stable vegetation |
| Positive | vegetation increase |

---

## Important Processing Step

The vegetation mask converts non-vegetated areas to NoData values.

Before subtraction:

```python
Con(IsNull(raster), 0, raster)
```

is used to convert NoData values to 0.

This allows the workflow to correctly detect vegetation disappearance between spring and summer periods.

---

## Output

Example outputs:

```text
NDVI_delta_2020.tif
NDVI_delta_2021.tif
```


## 1️⃣ Import Libraries

Import ArcPy, OS utilities, and Spatial Analyst tools required for raster calculations.


In [ ]:
import arcpy
import os
from arcpy.sa import *

# Enable Spatial Analyst extension
arcpy.CheckOutExtension("Spatial")

# Allow overwriting outputs
arcpy.env.overwriteOutput = True

## 2️⃣ Define Input and Output Folders

Specify the vegetation mask input folder and the output folder for NDVI delta rasters.


In [ ]:
# Update these paths before running the notebook
input_folder = r"path\to\input\NDVI_filtered"
output_folder = r"path\to\output\NDVI_delta"

# Create output folder if it does not exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

## 3️⃣ Read and Pair Seasonal Rasters

This step:

- Searches for vegetation rasters
- Detects spring and summer rasters
- Extracts years from filenames
- Pairs rasters for annual comparison


In [ ]:
# --------------------------------------------------
# Read raster files and pair them by year
# --------------------------------------------------

rasters = [
    f for f in os.listdir(input_folder)
    if f.lower().endswith((".tif", ".tiff"))
]

spring_rasters = {}
summer_rasters = {}

for f in rasters:

    name = f.lower()

    # Detect spring rasters
    if "spring" in name:

        year = ''.join([c for c in f if c.isdigit()])[:4]
        spring_rasters[year] = f

    # Detect summer rasters
    elif "summer" in name:

        year = ''.join([c for c in f if c.isdigit()])[:4]
        summer_rasters[year] = f

## 4️⃣ Calculate Seasonal NDVI Delta

For each year:

- Load spring and summer rasters
- Convert NoData values to 0
- Calculate NDVI difference
- Export annual delta rasters


In [ ]:
# --------------------------------------------------
# Calculate NDVI delta rasters
# --------------------------------------------------

for year in spring_rasters:

    if year in summer_rasters:

        spring_path = os.path.join(input_folder, spring_rasters[year])
        summer_path = os.path.join(input_folder, summer_rasters[year])

        print(f"Processing year: {year}")
        print("Spring raster:", spring_rasters[year])
        print("Summer raster:", summer_rasters[year])

        # Load rasters
        spring = Raster(spring_path)
        summer = Raster(summer_path)

        # Convert NoData values to 0
        spring_clean = Con(IsNull(spring), 0, spring)
        summer_clean = Con(IsNull(summer), 0, summer)

        # Calculate NDVI delta
        delta = summer_clean - spring_clean

        # Generate output filename
        out_name = f"NDVI_delta_{year}.tif"
        out_path = os.path.join(output_folder, out_name)

        # Save output raster
        delta.save(out_path)

        print("✔ Delta raster saved:", out_name)

    else:
        print(f"Missing summer raster for year: {year}")

print("\nNDVI delta calculation completed.")